# 🎬 AutoDub Studio — One-Click Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/Colab_Runner.ipynb)

ElevenLabs-quality **open-source automatic dubbing** — Demucs v4 · Pyannote 3.1 ·
SenseVoice-Small · CosyVoice 3.0 zero-shot cloning — wrapped in an iPhone-15
frosted-glass UI.

**How to use**
1. ▸ *Runtime* ▸ *Change runtime type* ▸ **T4 GPU** ▸ Save
2. ▸ *Runtime* ▸ **Run all**
3. When the last cell finishes, click the printed **`https://….gradio.live`** link
4. In Tab 1: upload your **audio/video master + Original SRT + Translated SRT**, hit 🔍
5. Tab 2: 🧬 matches speakers & emotions (paste a HuggingFace token in Tab 1 ▸ Advanced first — see [token guide](https://github.com/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-#-one-time-hugging-face-setup-required-for-step-3))
6. Tab 3: 🚀 renders the dubbed master mix / video

Ready to use — no build step required.

In [ ]:
# -- 0a > CLOUD PERSISTENCE SETUP (optional - default OFF) ---------------
# Modes: "none" (default, everything ephemeral) | "drive" | "hf"
# Cloud stores ONLY model caches (auto-cleanup); job files stay local.
PERSIST_MODE = "none"
HF_TOKEN_PERSIST = ""            # needed only for PERSIST_MODE = "hf"
AUTO_CLEANUP = True
import json as _json, pathlib as _pl
_cfg = {"mode": PERSIST_MODE, "hf_token": HF_TOKEN_PERSIST,
        "auto_cleanup": AUTO_CLEANUP}
_pcfg = _pl.Path("/content/autodub_persist.json")
try:
    import ipywidgets as _w
    from IPython.display import display as _d
    _tog = _w.Checkbox(value=(PERSIST_MODE != "none"),
                       description="Use cloud persistence (models only)")
    _mode = _w.Dropdown(options=["drive", "hf"], value="drive",
                        description="Backend")
    _tok = _w.Password(value=HF_TOKEN_PERSIST, description="HF token")
    _btn = _w.Button(description="Apply persistence settings")
    _out = _w.Output()
    def _apply_persist(_b):
        _cfg["mode"] = _mode.value if _tog.value else "none"
        _cfg["hf_token"] = _tok.value.strip()
        _pcfg.write_text(_json.dumps(_cfg), encoding="utf-8")
        with _out:
            print("saved - persistence mode:", _cfg["mode"])
    _btn.on_click(_apply_persist)
    _d(_w.VBox([_tog, _mode, _tok, _btn, _out]))
except Exception:
    pass
if not _pcfg.exists():
    _pcfg.write_text(_json.dumps(_cfg), encoding="utf-8")
print("persistence mode:",
      _json.loads(_pcfg.read_text(encoding="utf-8")).get("mode", "none"))

# ── 0 ▸ 🩺 DIAGNOSTICS — run me FIRST (and whenever anything breaks) ──
# Prints one copy-paste block: versions · GPU · disk/RAM · HF gate access.
import importlib.metadata as md, os, shutil, sys, urllib.request

print("═" * 64)
print(" 🩺 AUTODUB DIAGNOSTICS — paste this whole block with any bug report")
print("═" * 64)
print(f"python          {sys.version.split()[0]}")
print(f"platform        {sys.platform}")

_pkgs = ["torch", "torchaudio", "torchvision", "numpy", "scipy", "librosa",
         "soundfile", "gradio", "gradio-client", "huggingface-hub",
         "demucs", "pyannote-audio", "funasr", "modelscope", "pysrt",
         "moviepy", "pydub", "ffmpeg-python", "setuptools"]
for name in _pkgs:
    try:
        print(f"{name:16s} {md.version(name)}")
    except Exception:
        print(f"{name:16s} —")

try:
    import torch
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        print(f"gpu              {pr.name} · {pr.total_memory / 2**30:.0f} GB · CUDA {torch.version.cuda}")
    else:
        print("gpu              NONE — CPU mode (expected during Colab GPU cooldown)")
except Exception as e:
    print("gpu              probe failed:", e)

print(f"disk  /content   {shutil.disk_usage('/content').free / 2**30:.0f} GB free")
try:
    _m = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    print(f"ram              {_m / 2**30:.0f} GB")
except Exception:
    pass
print("ffmpeg           " + (shutil.which("ffmpeg") or "NOT FOUND"))

# ── HF gate probe (needs a token to be meaningful) ────────────────────
_token = (os.environ.get("HF_TOKEN") or "").strip()
if not _token and os.path.exists("/content/autodub_hf_token.txt"):
    _token = open("/content/autodub_hf_token.txt").read().strip()
if not _token:
    print("hf-gate          ⚠ no token found — paste it in Tab 1 ▸ Advanced, "
          "or set HF_TOKEN, then re-run this cell")
else:
    for repo in ("pyannote/speaker-diarization-3.1",
                 "pyannote/segmentation-3.0"):
        req = urllib.request.Request(
            f"https://huggingface.co/api/models/{repo}",
            headers={"Authorization": f"Bearer {_token}"})
        try:
            with urllib.request.urlopen(req, timeout=15) as r:
                print(f"hf-gate          ✅ {repo} accessible")
        except Exception as e:
            code = getattr(e, "code", None)
            if code in (401, 403):
                print(f"hf-gate          ❌ {repo} DENIED — open huggingface.co/{repo} "
                      "and click 'Agree and access repository' with the SAME account "
                      "that owns the token")
            else:
                print(f"hf-gate          ⚠ {repo} probe failed ({e}) — check network")
print("═" * 64)

In [ ]:
# -- 1 > Runtime check + PINNED ML-stack install (fail-soft, torch untouched) --
import importlib.metadata as md, subprocess, sys, torch

if torch.cuda.is_available():
    print(f"[OK] GPU ready: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)")
else:
    print("[!] No GPU on this VM - CPU mode (fine for Tabs 1-2 testing; "
          "full renders want the T4. Runtime > Change runtime type > T4 GPU).")

# Exact known-good pins - PROVEN to install on Colab py3.13 (verified live).
# torch / torchaudio / numpy / soundfile are deliberately NOT here: Colab's
# own builds must stay (GPU-CUDA wheel + system compatibility).
PINNED = {
    "demucs":          "4.0.1",
    "pyannote.audio":  "3.1.1",
    "funasr":          "1.2.6",
    "modelscope":      "1.27.1",
    "pysrt":           "1.1.2",
    "pydub":           "0.25.1",
    "moviepy":         "1.0.3",
    "ffmpeg-python":   "0.2.0",
    "librosa":         "0.11.0",
}
need = []
for pkg, ver in PINNED.items():
    try:
        have = md.version(pkg)
        ok = have == ver
    except Exception:
        have, ok = None, False
    status = ("[OK] " + str(have)) if ok else (str(have or "missing")
             + " -> will install " + ver)
    print(f"  {pkg:18s} {status}")
    if not ok:
        need.append(f"{pkg}=={ver}")

if need:
    print(f"\nInstalling {len(need)} pinned package(s) ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", *need],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("install done")
    else:
        # FAIL-SOFT: show the REAL pip error, keep 'Run all' going
        print("[!] pip INSTALL FAILED - tail of pip output:")
        for ln in (r.stdout or "").splitlines()[-25:]:
            print("   ", ln)
        for ln in (r.stderr or "").splitlines()[-10:]:
            print("   ", ln)
        print("[!] WARNING: ML stack incomplete - later steps may fail. "
              "Fix the error above and re-run this cell.")

# demucs.api (Separator) is MISSING from the PyPI 4.0.1 wheel - it only
# exists in the GitHub source. Install demucs from source when absent.
import importlib.util
if importlib.util.find_spec("demucs.api") is None:
    print("demucs.api missing from the PyPI wheel - installing demucs from source ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--no-deps",
                           "git+https://github.com/facebookresearch/demucs.git"])
    print("demucs source install done")

# verification imports - fail-soft (never halt 'Run all')
try:
    import demucs, pyannote, pysrt, funasr, modelscope  # noqa: F401
    from demucs.api import Separator  # noqa: F401
    print("[OK] ML stack verified: demucs / demucs.api / pyannote / funasr / modelscope / pysrt")
except Exception as e:
    print(f"[!] WARNING: stack verification failed ({e}) - "
          "re-run this cell after fixing the install error above.")

In [ ]:
# ── 2 ▸ Clone the project ─────────────────────────────────────────────
REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
!git clone {REPO_URL} ai-video-dubber
%cd ai-video-dubber
!ls

In [ ]:
# ── realign gradio-client to gradio's EXACT pin + full-stack audit ──
import importlib.metadata as md, os, subprocess, sys
try:
    g = md.version("gradio")
    want = next((r.split("==")[1].strip() for r in md.requires("gradio")
                 if r.startswith("gradio-client")), None)
    have = md.version("gradio_client")
    if want and have != want:
        print(f"gradio {g}: gradio-client {have} -> {want}")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "-q", f"gradio-client=={want}"])
    else:
        print(f"gradio {g} + gradio-client {have} already aligned")
except Exception as e:
    print("client realignment skipped:", e)

# audit: every remaining unpinned requirement from the project's file
_req = os.path.join(os.getcwd(), "requirements.txt")   # we are inside the clone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", _req])
print("✅ requirements.txt satisfied")

In [ ]:
# -- CosyVoice: AUTO-BOOTSTRAP v2 (self-clone + filtered deps + wget-style bar) --
import json, os, pathlib, re, site, subprocess, sys, time

CV = pathlib.Path("/content/CosyVoice")

# 1 - clone (with mirror fallback) if missing
if not (CV / "cosyvoice").is_dir():
    _urls = ["https://github.com/FunAudioLLM/CosyVoice.git",
             "https://github.com/QwenAudio/CosyVoice.git"]
    _ok = False
    for _u in _urls:
        print(f"Cloning CosyVoice from {_u} (shallow, with submodules) ...")
        t0 = time.time()
        rc = subprocess.call(["git", "clone", "--recursive", "--depth", "1",
                              "--shallow-submodules", "-q", _u, str(CV)])
        if rc == 0 and (CV / "cosyvoice").is_dir():
            print(f"cloned in {time.time() - t0:.0f}s")
            _ok = True
            break
        print(f"clone failed (rc={rc}) - trying next mirror ...")
        subprocess.call(["rm", "-rf", str(CV)])
    if not _ok:
        raise RuntimeError(
            "Could not clone CosyVoice from any known URL. Check network, or "
            "clone manually:\n  git clone --recursive "
            "https://github.com/FunAudioLLM/CosyVoice /content/CosyVoice")
else:
    print("OK /content/CosyVoice already present - skipping clone")

# 2 - register on sys.path (+ persist for every future process)
for p in (str(CV), str(CV / "third_party" / "Matcha-TTS")):
    if p not in sys.path:
        sys.path.insert(0, p)
pth = pathlib.Path(site.getsitepackages()[0]) / "autodub_cosyvoice.pth"
pth.write_text("\n".join([str(CV), str(CV / "third_party" / "Matcha-TTS")]),
               encoding="utf-8")

# 3 - FILTER Colab-hostile pins out of CosyVoice's requirements.
#     CosyVoice pins old torch/torchaudio/numpy/soundfile versions; several
#     have NO Python-3.13 wheels, which fails the ENTIRE pip transaction
#     (the exit-code-2 crash). Colab's own builds are newer and stay.
_src_req = CV / "requirements.txt"
_fil_req = CV / "requirements_colab.txt"
_BANNED = {"torch", "torchaudio", "torchvision", "numpy", "soundfile"}
_kept, _dropped = [], []
for _ln in _src_req.read_text(encoding="utf-8", errors="replace").splitlines():
    _s = _ln.strip()
    if not _s or _s.startswith("#"):
        continue
    if _s.startswith(("-", ".")) or "://" in _s:
        _kept.append(_ln)      # options / editable / vcs lines: keep verbatim
        continue
    _m = re.match(r"[A-Za-z0-9_.\-]+", _s)
    _raw = _m.group(0) if _m else ""
    if _raw.lower() in _BANNED:
        _dropped.append(_s)
    else:
        # strip version pins: old pins (e.g. onnxruntime==1.16.0) have no
        # py3.13 wheels and kill the whole transaction - install latest
        _kept.append(_raw + ("  # was: " + _s if _s != _raw else ""))
_fil_req.write_text("\n".join(_kept) + "\n", encoding="utf-8")
print(f"requirements filtered: {len(_kept)} kept / {len(_dropped)} dropped "
      f"(Colab-managed: {', '.join(_dropped) if _dropped else 'none'})")

# 4 - wget-style progress bar + ETA countdown
def _fmt_s(s):
    s = int(max(0, s)); m, sec = divmod(s, 60); h, m = divmod(m, 60)
    return f"{h}h{m:02d}m{sec:02d}s" if h else f"{m}m{sec:02d}s"

def _pip_count_targets(args):
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run",
                            "--quiet", "--report", "-", *args],
                           capture_output=True, text=True, timeout=900)
        return len(json.loads(r.stdout).get("install", []))
    except Exception:
        return None

def pip_progress(args, label="cosyvoice-reqs"):
    total = _pip_count_targets(args)
    if not total:
        total = sum(1 for l in _kept if l.strip())
    total = max(1, total)
    print(f"{label}: {total} package(s) to install - live progress below")
    proc = subprocess.Popen([sys.executable, "-m", "pip", "install",
                             "--disable-pip-version-check", "--progress-bar", "off",
                             *args],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    t0, seen, mb, lines = time.time(), set(), 0.0, []
    W = 30
    FULL = "\u2588" * W
    def draw(frac, note):
        done = int(W * frac)
        bar = "\u2588" * done + "\u2591" * (W - done)
        el = time.time() - t0
        eta = el / frac * (1 - frac) if frac > 0.02 else 0
        print(f"\r{label} [Elapsed {_fmt_s(el)} | ETA {_fmt_s(eta)}] "
              f"|{bar}| {frac*100:5.1f}%  {note}   ", end="", flush=True)
    dl_re = re.compile(r"Downloading\s+\S+\.whl\s+\(([\d.]+)\s*(MB|kB|GB|mB)\)",
                       re.IGNORECASE)
    for line in proc.stdout:
        line = line.strip()
        lines.append(line)
        m = re.match(r"Collecting\s+(\S+)", line)
        if m:
            seen.add(m.group(1).split("==")[0].split("[")[0].lower())
        dm = dl_re.search(line)
        if dm:
            mb += float(dm.group(1)) * {"kB": 0.001, "MB": 1, "mB": 1,
                                        "GB": 1024}[dm.group(2).lower()]
        note = f"{min(len(seen), total)}/{total} pkgs"
        if mb:
            note += f" \u00b7 {mb:.1f} MB \u00b7 {mb / max(1e-9, time.time()-t0):.1f} MB/s"
        draw(min(0.999, len(seen) / total), note)
    proc.wait()
    el = time.time() - t0
    if proc.returncode == 0:
        print(f"\r{label} [Elapsed {_fmt_s(el)}] |{FULL}| 100.0%  "
              f"done \u00b7 {mb:.1f} MB total   ")
    else:
        print(f"\r{label} [Elapsed {_fmt_s(el)}] FAILED "
              f"(exit {proc.returncode})          ")
        print("--- full pip output (last 40 lines) ---")
        for ln in lines[-40:]:
            print("   ", ln[:200])
        print("--- end of pip output ---")
        print("[!] WARNING: CosyVoice deps incomplete - Tab 3 rendering will "
              "need this fixed. 'Run all' continues.")

pip_progress(["-r", str(_fil_req)])

# 5 - sanity import (fail-soft so Run all continues)
try:
    import cosyvoice  # noqa: F401
    print("OK CosyVoice importable (source-tree mode, no pip install .)")
except Exception as e:
    print(f"[!] CosyVoice import failed: {str(e)[:200]}")
    print("    Rendering (Tab 3) will report exact bootstrap steps when run.")


# 6 - OPTIONAL: persist outputs/ + uploads/ (+ model caches) to Google Drive
#     so a Colab disconnect never loses pipeline progress. Set False to skip.
_persist = {"mode": "none", "hf_token": "", "auto_cleanup": True}
_pcfg = pathlib.Path("/content/autodub_persist.json")
if _pcfg.exists():
    try:
        _persist.update(json.loads(_pcfg.read_text(encoding="utf-8")))
    except Exception:
        pass
PERSIST_TO_DRIVE = (_persist.get("mode") == "drive")
import shutil as _sh
if PERSIST_TO_DRIVE:
    try:
        from google.colab import drive as _drive
        if not os.path.exists("/content/drive/MyDrive"):
            _drive.mount("/content/drive")
        _drv = pathlib.Path("/content/drive/MyDrive/AutoDub_Studio")
        _drv.mkdir(parents=True, exist_ok=True)
        _proj = pathlib.Path.cwd()          # the clone dir (Cell 3 cd'd here)
        for _name in ("outputs", "uploads"):
            _src, _dst = _proj / _name, _drv / _name
            _dst.mkdir(parents=True, exist_ok=True)
            if _src.exists() and not _src.is_symlink():
                for _f in _src.iterdir():
                    if not (_dst / _f.name).exists():
                        _sh.move(str(_f), str(_dst / _f.name))
                _sh.rmtree(_src, ignore_errors=True)
            if not _src.is_symlink():
                os.symlink(str(_dst), str(_src))
        if True:  # auto-cleanup: cloud stores models only
            _mc = _drv / "model_cache"
            _mc.mkdir(exist_ok=True)
            os.environ.setdefault("MODELSCOPE_CACHE", str(_mc / "modelscope"))
            os.environ.setdefault("HF_HOME", str(_mc / "huggingface"))
        print("OK outputs/ + uploads/ persist to Google Drive "
              "(MyDrive/AutoDub_Studio) - disconnects no longer lose progress"
              + (" + model caches" if PERSIST_MODEL_CACHE else ""))
    except Exception as e:
        print("[!] Drive persistence skipped:", str(e)[:160])

# 6b - HF-Hub persistence backend (models only; job files stay local)
if _persist.get("mode") == "hf" and _persist.get("hf_token"):
    import shutil as _sh
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=_persist["hf_token"])
        _user = _api.whoami().get("name", "user")
        _repo = f"{_user}/autodub-model-cache"
        _persist["hf_repo"] = _repo
        _pcfg = pathlib.Path("/content/autodub_persist.json")
        _pcfg.write_text(json.dumps(_persist), encoding="utf-8")
        try:
            _api.create_repo(_repo, repo_type="dataset", private=True,
                             exist_ok=True)
        except Exception:
            pass
        try:
            from huggingface_hub import snapshot_download as _hfsnap
            _hfsnap(_repo, repo_type="dataset",
                    local_dir="/root/.cache/autodub_pull")
            _src = pathlib.Path("/root/.cache/autodub_pull")
            if _src.exists():
                for _f in _src.rglob("*"):
                    if _f.is_file():
                        _rel = _f.relative_to(_src)
                        _dst = pathlib.Path("/root/.cache") / _rel
                        _dst.parent.mkdir(parents=True, exist_ok=True)
                        if not _dst.exists():
                            _sh.copy2(str(_f), str(_dst))
                print("OK model cache restored from HF Hub")
        except Exception as e:
            print("[i] HF cache pull skipped:", str(e)[:120])
    except Exception as e:
        print("[!] HF persistence setup failed:", str(e)[:160])

# 7 - PRE-DOWNLOAD all model checkpoints NOW (before GPU work) so Tabs 2-3
#     never stall on slow downloads while the GPU sits idle.
print("Pre-downloading model checkpoints (one-time, then cached) ...")
try:
    from modelscope import snapshot_download as _msdl
    for _repo in ("iic/SenseVoiceSmall", "iic/CosyVoice2-0.5B"):
        try:
            _msdl(_repo)
            print(f"  cached: {_repo}")
        except Exception as e:
            print(f"  [!] {_repo}: {str(e)[:120]}")
except Exception as e:
    print("  [!] modelscope pre-download skipped:", str(e)[:120])

_tok = (os.environ.get("HF_TOKEN") or "").strip()
if not _tok and os.path.exists("/content/autodub_hf_token.txt"):
    _tok = open("/content/autodub_hf_token.txt").read().strip()
if _tok:
    try:
        from huggingface_hub import snapshot_download as _hsdl
        for _repo in ("pyannote/speaker-diarization-3.1",
                      "pyannote/segmentation-3.0"):
            try:
                _hsdl(_repo, token=_tok)
                print(f"  cached: {_repo}")
            except Exception as e:
                print(f"  [!] {_repo}: {str(e)[:120]}")
    except Exception as e:
        print("  [!] pyannote pre-download skipped:", str(e)[:120])
else:
    print("  (i) no HF token found yet - pyannote will download during Tab 2")


# 8 - push model caches to HF Hub (hf mode only; models-only auto-cleanup)
if _persist.get("mode") == "hf" and _persist.get("hf_token") \
        and _persist.get("hf_repo"):
    import shutil as _sh
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=_persist["hf_token"])
        _stage = pathlib.Path("/root/.cache/autodub_stage")
        _sh.rmtree(_stage, ignore_errors=True)
        for _pref, _src in (("modelscope", "/root/.cache/modelscope"),
                            ("hf", "/root/.cache/huggingface")):
            _s = pathlib.Path(_src)
            if _s.exists():
                (_stage / _pref).mkdir(parents=True, exist_ok=True)
                for _f in _s.rglob("*"):
                    if _f.is_file() and _f.suffix != ".lock":
                        _rel = _f.relative_to(_s)
                        _d = _stage / _pref / _rel
                        _d.parent.mkdir(parents=True, exist_ok=True)
                        if not _d.exists():
                            _sh.copy2(str(_f), str(_d))
        _api.upload_folder(folder_path=str(_stage),
                           repo_id=_persist["hf_repo"],
                           repo_type="dataset",
                           commit_message="AutoDub model cache sync")
        print("OK model cache pushed to HF Hub:", _persist["hf_repo"])
    except Exception as e:
        print("[!] HF cache push skipped:", str(e)[:160])

In [ ]:
# ── 5b ▸ 🔄 OPTIONAL · Sync model cache → HF Hub (manual, resumable) ──
# Run this cell ONLY when you want to push the VM's downloaded models to
# your private HF dataset repo (e.g. models arrived late, or an upload was
# interrupted). Safe to re-run — resumable, models-only, never blocks.
HF_TOKEN_SYNC = ""    # optional: fill in to override the saved config token

import json as _json, os as _os, pathlib as _pl, shutil as _sh, time as _time

_cfg_p = _pl.Path("/content/autodub_persist.json")
_cfg = {}
if _cfg_p.exists():
    try:
        _cfg = _json.loads(_cfg_p.read_text(encoding="utf-8"))
    except Exception:
        pass
TOKEN = HF_TOKEN_SYNC or _cfg.get("hf_token") or _os.environ.get("HF_TOKEN", "")
MODE = _cfg.get("mode", "none")
if MODE == "hf" and _cfg.get("synced_this_session"):
    print("✅ already synced this session — nothing to do (delete "
          "/content/autodub_persist.json 'synced_this_session' key to force)")
else:
    if not TOKEN:
        raise SystemExit("No token: fill HF_TOKEN_SYNC above, or run Cell 0 "
                         "persistence setup with mode=hf first.")
    from huggingface_hub import HfApi
    _api = HfApi(token=TOKEN)
    _user = _api.whoami().get("name", "user")
    REPO = _cfg.get("hf_repo") or f"{_user}/autodub-model-cache"
    print("HF user:", _user, "· repo:", REPO)
    try:
        _api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
        print("repo ready (private dataset)")
    except Exception as e:
        print("repo setup:", str(e)[:120])

    stage = _pl.Path("/root/.cache/autodub_stage")
    _sh.rmtree(stage, ignore_errors=True)
    SOURCES = {"modelscope": "/root/.cache/modelscope",
               "huggingface": "/root/.cache/huggingface"}
    t0, n = _time.time(), 0
    for pref, src in SOURCES.items():
        s = _pl.Path(src)
        if not s.exists():
            print("  (no cache at", src, ")")
            continue
        for f in s.rglob("*"):
            if f.is_file() and f.suffix != ".lock" and "tmp" not in f.name:
                dst = stage / pref / f.relative_to(s)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists():
                    _sh.copy2(str(f), str(dst))
                    n += 1
    print(f"staged {n} file(s) in {_time.time()-t0:.0f}s")

    print("uploading (resumable — re-run continues where it stopped) ...")
    _api.upload_folder(folder_path=str(stage), repo_id=REPO,
                       repo_type="dataset",
                       commit_message="AutoDub model cache sync")
    print("✅ model cache pushed to:", REPO)

    _cfg.update({"mode": "hf", "hf_token": TOKEN, "hf_repo": REPO,
                 "synced_this_session": True, "auto_cleanup": True})
    _cfg_p.write_text(_json.dumps(_cfg), encoding="utf-8")
    print("✅ persistence config saved (mode=hf) — future sessions auto-restore")

In [ ]:
# ── 5 ▸ 🚀 LAUNCH — click the https://….gradio.live link printed below ─
!python app.py